# Discount Profitability ML

**Research questions**
1. How do historically observed discounting practices relate to profit margins across product categories?
2. Do non-linear models outperform linear models in capturing discount-profit relationships?

The main modelling task uses discounted transactions and predicts `profit_margin`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent
if not (ROOT / "src").exists():
    raise RuntimeError("Could not locate repository root containing src/")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.inspection import PartialDependenceDisplay
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

from src.preprocessing import prepare_model_data, create_features
from src.train import build_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor

In [ ]:
DATA_PATH = ROOT / "data" / "ecommerce_sales_34500.csv"
df = pd.read_csv(DATA_PATH)
df.info()
df.describe(include="all")

## 1. Exploratory Data Analysis


In [ ]:
fig, ax = plt.subplots()
ax.hist(df["discount"], bins=30)
ax.set_xlabel("Discount Rate")
ax.set_ylabel("Frequency")
ax.set_title("Distribution of Applied Discount Rates")
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.hist(df["profit_margin"], bins=30)
ax.set_xlabel("Profit Margin")
ax.set_ylabel("Frequency")
ax.set_title("Distribution of Transaction-Level Profit Margins")
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.scatter(df["discount"], df["profit_margin"], alpha=0.3)
ax.set_xlabel("Discount Rate")
ax.set_ylabel("Profit Margin")
ax.set_title("Discount Rate vs Profit Margin")
plt.show()

In [ ]:
df_discounted = df[df["discount"] > 0].copy()
fig, ax = plt.subplots()
df_discounted.boxplot(column="profit_margin", by="category", rot=45, ax=ax)
ax.set_xlabel("Product Category")
ax.set_ylabel("Profit Margin")
ax.set_title("Profit Margin by Category (Discounted Transactions)")
plt.suptitle("")
plt.show()

## 2. Feature Engineering

The project found that discount percentage and category alone had weak explanatory power. Two engineered variables were therefore introduced:

- `discount_value = (price × quantity) × discount`
- `unit_revenue = (total_amount − shipping_cost) / quantity`

`unit_revenue` is treated as a proxy because the dataset does not contain explicit COGS.

In [ ]:
engineered = create_features(df_discounted)

fig, ax = plt.subplots()
ax.scatter(engineered["unit_revenue"], engineered["profit_margin"], alpha=0.3)
ax.set_xlabel("Unit Revenue")
ax.set_ylabel("Profit Margin")
ax.set_title("Unit Revenue vs Profit Margin")
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.scatter(engineered["discount_value"], engineered["profit_margin"], alpha=0.3)
ax.set_xlabel("Discount Value")
ax.set_ylabel("Profit Margin")
ax.set_title("Discount Value vs Profit Margin")
plt.show()

## 3. Train/Test Split and Model Comparison

Both models use the same 80/20 split and random seed of 42. 

In [ ]:
X, y = prepare_model_data(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

linear_model = build_pipeline(LinearRegression(), drop_first=True)
gbr_model = build_pipeline(GradientBoostingRegressor(random_state=42), drop_first=False)

linear_model.fit(X_train, y_train)
gbr_model.fit(X_train, y_train)

linear_pred = linear_model.predict(X_test)
gbr_pred = gbr_model.predict(X_test)

def metrics(y_true, pred):
    return {
        "R2": r2_score(y_true, pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, pred)),
        "MAE": mean_absolute_error(y_true, pred),
    }

results = pd.DataFrame([
    {"Model": "Linear Regression", **metrics(y_test, linear_pred)},
    {"Model": "Gradient Boosting", **metrics(y_test, gbr_pred)},
])
results

### Reported research result

The submitted report reports R² 0.6626 / RMSE 25.94 / MAE 13.60 for Linear Regression and R² 0.7765 / RMSE 21.11 / MAE 9.32 for GBR. 

## 4. Linear Regression Coefficients

In [ ]:
linear_preprocessor = linear_model.named_steps["preprocessor"]
linear_estimator = linear_model.named_steps["model"]
linear_features = linear_preprocessor.get_feature_names_out()

coefficients = pd.DataFrame({
    "Feature": linear_features,
    "Coefficient": linear_estimator.coef_,
}).sort_values("Coefficient", ascending=False)
coefficients

## 5. Gradient Boosting Feature Importance

In [ ]:
gbr_preprocessor = gbr_model.named_steps["preprocessor"]
gbr_estimator = gbr_model.named_steps["model"]
gbr_features = gbr_preprocessor.get_feature_names_out()

importance = pd.DataFrame({
    "Feature": gbr_features,
    "Importance": gbr_estimator.feature_importances_,
}).sort_values("Importance", ascending=False)
importance

## 6. Partial Dependence

The original analysis used partial dependence to examine the marginal behaviour of discount value and unit revenue.

In [ ]:
X_test_transformed = gbr_preprocessor.transform(X_test)
transformed_names = list(gbr_features)
feature_indices = [transformed_names.index("numeric__discount_value"), transformed_names.index("numeric__unit_revenue")]

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
PartialDependenceDisplay.from_estimator(
    gbr_estimator,
    X_test_transformed,
    feature_indices,
    feature_names=transformed_names,
    grid_resolution=50,
    ax=ax,
)
plt.tight_layout()
plt.show()

## 7. SHAP Model Interpretation

In [ ]:
import shap

explainer = shap.TreeExplainer(gbr_estimator)
shap_values = explainer.shap_values(X_test_transformed)
shap.summary_plot(
    shap_values,
    X_test_transformed,
    feature_names=transformed_names,
)

In [ ]:
shap.dependence_plot(
    transformed_names.index("numeric__discount_value"),
    shap_values,
    X_test_transformed,
    feature_names=transformed_names,
)

## 8. Discount Sensitivity Simulation

The simulation follows the research design: quantity is fixed at one, a baseline pre-discount price is inferred from training-data averages, discount rate varies from 1% to 50%, and category is held constant while discount value and unit revenue are updated.

In [ ]:
discount_grid = np.linspace(0.01, 0.50, 50)
quantity = 1
baseline_unit_revenue = X_train["unit_revenue"].mean()
baseline_discount = X_train["discount_value"].mean() / (
    X_train["unit_revenue"].mean() + X_train["discount_value"].mean()
)
base_price = baseline_unit_revenue / (1 - baseline_discount)

def simulation_frame(category):
    rows = []
    for d in discount_grid:
        rows.append({
            "discount_value": base_price * d * quantity,
            "unit_revenue": base_price * (1 - d),
            "category": category,
        })
    return pd.DataFrame(rows)

category_values = sorted(df_discounted["category"].unique())
category = category_values[0]
sim = simulation_frame(category)
sim["pred_margin_lr"] = linear_model.predict(sim)
sim["pred_margin_gbr"] = gbr_model.predict(sim)

fig, ax = plt.subplots()
ax.plot(discount_grid * 100, sim["pred_margin_lr"], label="Linear Regression")
ax.plot(discount_grid * 100, sim["pred_margin_gbr"], label="Gradient Boosting")
ax.set_xlabel("Discount Percentage (%)")
ax.set_ylabel("Predicted Profit Margin")
ax.set_title(f"Discount Sensitivity: {category}")
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for category in category_values:
    sim = simulation_frame(category)
    sim["pred_margin_gbr"] = gbr_model.predict(sim)
    ax.plot(discount_grid * 100, sim["pred_margin_gbr"], label=category)

ax.set_xlabel("Discount Percentage (%)")
ax.set_ylabel("Predicted Profit Margin")
ax.set_title("GBR Discount Sensitivity by Category")
ax.legend()
plt.show()

## 9. Residual Diagnostics

In [ ]:
discount_pct_test = (
    X_test["discount_value"] /
    (X_test["unit_revenue"] + X_test["discount_value"])
) * 100

residuals_lr = y_test - linear_pred
residuals_gbr = y_test - gbr_pred

fig, ax = plt.subplots()
ax.scatter(discount_pct_test, residuals_lr, alpha=0.4, label="Linear Regression", s=20)
ax.scatter(discount_pct_test, residuals_gbr, alpha=0.4, label="Gradient Boosting", s=20)
ax.axhline(0, linestyle="--")
ax.set_xlabel("Discount Percentage (%)")
ax.set_ylabel("Residual (Actual − Predicted Margin)")
ax.set_title("Residuals vs Discount Level")
ax.legend()
plt.show()

## 10. Conclusions and Limitations

The research found that the non-linear GBR model outperformed the linear baseline and that transaction-scale features were more influential than most category indicators. The model behaviour should be interpreted as historical association, not causal evidence that higher discounts increase profitability.

Key limitations retained from the submitted work:
- historical discounts are not randomly assigned;
- explicit COGS is unavailable;
- focusing on discounted transactions introduces selection bias;
- the simulation does not model demand elasticity or customer behaviour;
- SHAP/PDP explain model behaviour rather than the underlying economic process.